# Project UAS Pemrograman 3: Apache Airflow ETL & Monitoring Pipeline
**Dataset:** Open-Meteo Weather API (Koordinat Surabaya: `-7.25, 112.75`)  
**Arsitektur:** Multi-DAG Orchestration (`dag_1_etl_cuaca` $\rightarrow$ `dag_2_ringkasan_monitoring`)

Notebook ini dirancang untuk men-generate dua file DAG Apache Airflow yang mencakup keseluruhan cakupan modul:
1. **Desain Workflow & ETL (`dag_1_etl_cuaca.py`):**
   - **Extract:** Mengambil data cuaca per jam (suhu, kelembapan, curah hujan, kecepatan angin) dari API Open-Meteo.
   - **Transform:** Membersihkan data null, konversi waktu, dan kategorisasi kondisi cuaca menggunakan `pandas`.
   - **Load:** Menyimpan data bersih ke CSV (`weather_surabaya_clean.csv`).
   - **Inter-DAG Trigger:** Memicu DAG ke-2 secara otomatis menggunakan `TriggerDagRunOperator`.
2. **Monitoring, Analytics & Error Handling (`dag_2_ringkasan_monitoring.py`):**
   - Membaca hasil CSV dari DAG 1, membuat laporan ringkasan harian (`weather_surabaya_summary.csv`).
   - Dilengkapi `retries`, `on_failure_callback`, dan task `alert_error_handler` (`trigger_rule='one_failed'`) untuk simulasi monitoring.

## 1. Persiapan Direktori & Dependency
Jalankan sel ini di lingkungan Linux / VM tempat Apache Airflow terpasang.

In [ ]:
%%bash
pip install -q requests pandas
mkdir -p ~/airflow/dags/output_weather
echo "Direktori output siap di ~/airflow/dags/output_weather"

## 2. Generate DAG 1: Main ETL Pipeline & Trigger (`dag_1_etl_cuaca.py`)
DAG ini bertugas menarik data API Open-Meteo, melakukan transformasi dengan `pandas`, menyimpan ke CSV, lalu memicu `dag_2_ringkasan_monitoring`.

In [ ]:
%%writefile ~/airflow/dags/dag_1_etl_cuaca.py
from datetime import datetime, timedelta
import os
import logging
import requests
import pandas as pd
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.trigger_dagrun import TriggerDagRunOperator

BASE_DIR = os.path.expanduser("~/airflow/dags/output_weather")
RAW_JSON_PATH = os.path.join(BASE_DIR, "raw_weather.json")
CLEAN_CSV_PATH = os.path.join(BASE_DIR, "weather_surabaya_clean.csv")

def on_fail_callback(context):
    task_id = context["task_instance"].task_id
    logging.error(f"[ALERT MONITORING] Task '{task_id}' pada DAG 1 gagal dieksekusi!")

def extract_weather_api():
    os.makedirs(BASE_DIR, exist_ok=True)
    url = (
        "https://api.open-meteo.com/v1/forecast"
        "?latitude=-7.25&longitude=112.75"
        "&hourly=temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m"
        "&timezone=Asia%2FJakarta&forecast_days=1"
    )
    logging.info(f"Mengambil data cuaca Surabaya dari: {url}")
    response = requests.get(url, timeout=15)
    response.raise_for_status()
    
    with open(RAW_JSON_PATH, "w", encoding="utf-8") as f:
        f.write(response.text)
    logging.info(f"Extract sukses! Raw JSON disimpan di {RAW_JSON_PATH}")
    return RAW_JSON_PATH

def transform_weather_data(ti):
    raw_path = ti.xcom_pull(task_ids="extract_weather_api")
    raw_data = pd.read_json(raw_path)
    hourly = raw_data["hourly"]
    
    df = pd.DataFrame({
        "waktu_wib": hourly["time"],
        "suhu_c": hourly["temperature_2m"],
        "kelembapan_persen": hourly["relative_humidity_2m"],
        "curah_hujan_mm": hourly["precipitation"],
        "kecepatan_angin_kmh": hourly["wind_speed_10m"]
    })
    
    # Validasi kualitas data (Data Quality Check)
    if df["suhu_c"].isnull().any() or (df["suhu_c"] > 55).any():
        raise ValueError("Data Quality Error: Ditemukan nilai suhu tidak valid!")
    
    # Feature Engineering sederhana
    df["kondisi_cuaca"] = df["curah_hujan_mm"].apply(
        lambda x: "Hujan" if x > 0 else "Cerah / Berawan"
    )
    df["waktu_ekstraksi"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    temp_clean_path = os.path.join(BASE_DIR, "temp_clean.csv")
    df.to_csv(temp_clean_path, index=False)
    logging.info(f"Transform sukses! Total {len(df)} baris data bersih siap di-load.")
    return temp_clean_path

def load_to_storage(ti):
    temp_path = ti.xcom_pull(task_ids="transform_weather_data")
    df_new = pd.read_csv(temp_path)
    
    # Append jika file sudah ada, buat baru jika belum ada
    file_exists = os.path.isfile(CLEAN_CSV_PATH)
    df_new.to_csv(CLEAN_CSV_PATH, mode="a", header=not file_exists, index=False)
    logging.info(f"Load sukses! Data tersimpan ke {CLEAN_CSV_PATH}")

default_args = {
    "owner": "tim_data_sains",
    "depends_on_past": False,
    "retries": 2,
    "retry_delay": timedelta(seconds=15),
    "on_failure_callback": on_fail_callback,
}

with DAG(
    dag_id="dag_1_etl_cuaca_surabaya",
    default_args=default_args,
    description="Pipeline ETL Utama Open-Meteo Surabaya + Inter-DAG Trigger",
    schedule="@daily",
    start_date=datetime(2026, 11, 1),
    catchup=False,
    tags=["uas", "etl", "open-meteo"],
) as dag:
    
    t_extract = PythonOperator(
        task_id="extract_weather_api",
        python_callable=extract_weather_api,
    )
    
    t_transform = PythonOperator(
        task_id="transform_weather_data",
        python_callable=transform_weather_data,
    )
    
    t_load = PythonOperator(
        task_id="load_to_storage",
        python_callable=load_to_storage,
    )
    
    t_trigger_dag2 = TriggerDagRunOperator(
        task_id="trigger_dag_ringkasan",
        trigger_dag_id="dag_2_ringkasan_monitoring",
        wait_for_completion=False,
    )
    
    t_extract >> t_transform >> t_load >> t_trigger_dag2

## 3. Generate DAG 2: Downstream Analytics & Error Monitoring (`dag_2_ringkasan_monitoring.py`)
DAG ini dipicu otomatis oleh DAG 1, bertugas menghitung statistik cuaca harian dan menyediakan jalur penanganan error (`one_failed`).

In [ ]:
%%writefile ~/airflow/dags/dag_2_ringkasan_monitoring.py
from datetime import datetime, timedelta
import os
import logging
import pandas as pd
from airflow import DAG
from airflow.operators.python import PythonOperator

BASE_DIR = os.path.expanduser("~/airflow/dags/output_weather")
CLEAN_CSV_PATH = os.path.join(BASE_DIR, "weather_surabaya_clean.csv")
SUMMARY_CSV_PATH = os.path.join(BASE_DIR, "weather_surabaya_summary.csv")

def generate_daily_summary():
    if not os.path.exists(CLEAN_CSV_PATH):
        raise FileNotFoundError(f"File sumber {CLEAN_CSV_PATH} tidak ditemukan! Jalankan DAG 1 terlebih dahulu.")
    
    df = pd.read_csv(CLEAN_CSV_PATH)
    summary = pd.DataFrame([{
        "tanggal_laporan": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "total_record": len(df),
        "suhu_rata_rata_c": round(df["suhu_c"].mean(), 2),
        "suhu_maksimum_c": df["suhu_c"].max(),
        "suhu_minimum_c": df["suhu_c"].min(),
        "rata_rata_kelembapan": round(df["kelembapan_persen"].mean(), 2),
        "total_jam_hujan": int((df["kondisi_cuaca"] == "Hujan").sum())
    }])
    
    file_exists = os.path.isfile(SUMMARY_CSV_PATH)
    summary.to_csv(SUMMARY_CSV_PATH, mode="a", header=not file_exists, index=False)
    logging.info(f"Laporan ringkasan tersimpan di {SUMMARY_CSV_PATH}")
    logging.info(f"Statistik Terkini:\n{summary.to_string(index=False)}")

def pipeline_success_log():
    logging.info("SELURUH PIPELINE ETL & MONITORING SELESAI DENGAN SUKSES.")

def alert_error_handler():
    logging.error("PEMULIHAN AKTIF: Terdeteksi kegagalan pada task summary. Mengirim log insiden ke administrator.")

default_args = {
    "owner": "tim_monitoring",
    "retries": 1,
    "retry_delay": timedelta(seconds=10),
}

with DAG(
    dag_id="dag_2_ringkasan_monitoring",
    default_args=default_args,
    description="DAG Downstream untuk Agregasi Statistik & Error Handling",
    schedule=None,  # Hanya berjalan ketika dipicu oleh DAG 1
    start_date=datetime(2026, 11, 1),
    catchup=False,
    tags=["uas", "monitoring", "trigger"],
) as dag:
    
    t_summary = PythonOperator(
        task_id="generate_daily_summary",
        python_callable=generate_daily_summary,
    )
    
    t_success = PythonOperator(
        task_id="pipeline_success_log",
        python_callable=pipeline_success_log,
        trigger_rule="all_success",
    )
    
    t_error = PythonOperator(
        task_id="alert_error_handler",
        python_callable=alert_error_handler,
        trigger_rule="one_failed",
    )
    
    t_summary >> [t_success, t_error]

## 4. Sinkronisasi DAG & Uji Eksekusi CLI
Jalankan perintah di bawah ini untuk memverifikasi bahwa kedua DAG sudah terbaca oleh Airflow tanpa error sintaks.

In [ ]:
%%bash
airflow dags reserialize
airflow dags list | grep dag_

## 5. Verifikasi Hasil Output CSV (Untuk Screenshot Laporan Bagian 5)
Setelah menjalankan `dag_1_etl_cuaca_surabaya` di Airflow UI (pastikan toggle DAG dalam keadaan **Unpaused / On**), jalankan sel ini untuk menampilkan tabel output yang rapi untuk kebutuhan dokumentasi.

In [ ]:
import os
import pandas as pd

base_dir = os.path.expanduser("~/airflow/dags/output_weather")
clean_file = os.path.join(base_dir, "weather_surabaya_clean.csv")
summary_file = os.path.join(base_dir, "weather_surabaya_summary.csv")

if os.path.exists(clean_file):
    print("=== HASIL LOAD DATA BERSIH (10 BARIS PERTAMA) ===")
    display(pd.read_csv(clean_file).head(10))
else:
    print("File clean CSV belum tersedia. Jalankan DAG 1 di Airflow UI terlebih dahulu.")

if os.path.exists(summary_file):
    print("\n=== HASIL RINGKASAN HARIAN (DAG 2) ===")
    display(pd.read_csv(summary_file))
else:
    print("File summary CSV belum tersedia.")